In [ ]:
# #!/usr/bin/env python3
# import os
# import re
# import glob
# import rasterio
# import xarray as xr
# import pandas as pd
# import numpy as np
# import logging
# from shapely.geometry import box
# from datetime import datetime
# from rasterio.warp import reproject, Resampling
# from concurrent.futures import ProcessPoolExecutor, as_completed
# import time
# import geopandas as gpd
# import sys
# import argparse

# # --------------------------
# # Configuration
# # --------------------------
# CRS_TARGET = "EPSG:27700"
# DEM_DIR = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/Model_builds/Pluvial/v4/dem/"
# CSV_DIR = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/UKCP_rainfall_events/fixed_threshold_30mm/"
# NETCDF_DIR = "/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/"
# SHAPEFILE_PATH = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/CEH_catchments/CEH_IHU_with_coastline/hyd_areas_GB_with_subcatchments.shp"
# SOIL_TEXTURE_DIR = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/Model_builds/Pluvial/v4/soil_texture/"
# POROSITY_CSV = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/SoilTexture/USDA_soil_texture_effective_porosity.csv"

# OUTPUT_ORIG_TIF_DIR = "/scratch/hydro5/users/la17355/FUTURE-FLOOD/UKCP_soil_moisture/original_grids/"
# OUTPUT_VOL_DIR = "/scratch/hydro5/users/la17355/FUTURE-FLOOD/UKCP_soil_moisture/volumetric_30m/"

# LOG_DIR = "/scratch/hydro5/users/la17355/FUTURE-FLOOD/UKCP_soil_moisture/"

# DRY_RUN = False
# MAX_WORKERS = min(8, os.cpu_count())

# # --------------------------
# # Helper: robust HA extraction
# # --------------------------
# def extract_ha_num(path_or_name):
#     """
#     Extract full HA identifier from a filename or basename.
#     Matches:
#       - 27
#       - 27_a
#       - 54_d
#       - 33_b
#     Returns the matched token (string) or None if not found.
#     """
#     base = os.path.splitext(os.path.basename(path_or_name))[0]
#     # look for a trailing token that starts with digits and may have a _suffix (letters or digits)
#     m = re.search(r'(\d+(?:_[A-Za-z0-9]+)?)$', base)
#     return m.group(1) if m else None

# # --------------------------
# # Utility Functions
# # --------------------------
# def load_catchments(shapefile_path):
#     return gpd.read_file(shapefile_path)

# def find_matching_netcdf(netcdf_dir, year, month, day):
#     target_date_int = int(f"{year:04d}{month:02d}{day:02d}")
#     for nc_file in glob.glob(os.path.join(netcdf_dir, "*.nc")):
#         match = re.search(r"_(\d{8})-(\d{8})_", os.path.basename(nc_file))
#         if match and int(match.group(1)) <= target_date_int <= int(match.group(2)):
#             return nc_file
#     logging.warning(f"[NetCDF Search Fail] No match for date {target_date_int} in {netcdf_dir}")
#     return None

# def export_gravitational(ds, var, out_path, bounds):
#     if not ds.rio.crs:
#         ds = ds.rio.write_crs(CRS_TARGET)
#     clipped = ds[[var]].rio.clip_box(
#         minx=bounds.bounds[0],
#         miny=bounds.bounds[1],
#         maxx=bounds.bounds[2],
#         maxy=bounds.bounds[3]
#     )
#     os.makedirs(os.path.dirname(out_path), exist_ok=True)
#     clipped[var].rio.to_raster(out_path, compress="LZW", tiled=True)

# def resample_to_30m_array(in_path, dem):
#     with rasterio.open(in_path) as src:
#         out_shape = (src.count, dem["height"], dem["width"])
#         resampled = np.empty(out_shape, dtype=src.dtypes[0])

#         for i in range(1, src.count + 1):
#             reproject(
#                 source=rasterio.band(src, i),
#                 destination=resampled[i - 1],
#                 src_transform=src.transform,
#                 src_crs=src.crs,
#                 dst_transform=dem["transform"],
#                 dst_crs=dem["crs"],
#                 resampling=Resampling.nearest
#             )

#         out_meta = src.meta.copy()
#         out_meta.update({
#             'crs': dem["crs"],
#             'transform': dem["transform"],
#             'width': dem["width"],
#             'height': dem["height"],
#             'compress': 'LZW',
#             'tiled': True,
#             'nodata': -9999,
#             'dtype': src.dtypes[0]
#         })

#         return resampled[0], out_meta

# def convert_to_volumetric_array(grav_data, grav_meta, vol_tif_out, vol_asc_out, porosity_data, dem_mask):
#     try:
#         dem_mask_bool = (dem_mask == -9999)

#         with np.errstate(divide="ignore", invalid="ignore"):
#             vol_sm = (grav_data / 1000.0) / porosity_data
#             vol_sm = np.clip(vol_sm, 0, 1)
#             vol_sm = np.round(vol_sm, 2)

#         vol_sm[dem_mask_bool] = -9999

#         grav_meta.update(dtype=rasterio.float32, nodata=-9999, compress='LZW', driver='GTiff')

#         os.makedirs(os.path.dirname(vol_tif_out), exist_ok=True)
#         with rasterio.open(vol_tif_out, 'w', **grav_meta) as dst:
#             dst.write(vol_sm.astype(np.float32), 1)

#         asc_meta = grav_meta.copy()
#         asc_meta.pop('compress', None)
#         asc_meta.pop('tiled', None)
#         asc_meta.update(driver='AAIGrid')

#         os.makedirs(os.path.dirname(vol_asc_out), exist_ok=True)
#         with rasterio.open(vol_asc_out, 'w', **asc_meta) as dst:
#             dst.write(vol_sm.astype(np.float32), 1)

#     except Exception as e:
#         logging.error(f"[Volumetric Conversion Error] {vol_tif_out}: {e}")

# def load_porosity_grids(POROSITY_CSV, SOIL_TEXTURE_DIR, TARGET_HAS):
#     df = pd.read_csv(POROSITY_CSV)
#     porosity_map = df.set_index("USDA_soil")["Effective_porosity"].to_dict()
#     soil_grids = {}
#     for tif in glob.glob(os.path.join(SOIL_TEXTURE_DIR, "*.tif")):
#         ha_num = extract_ha_num(tif)
#         if ha_num is None:
#             continue
#         if ha_num not in TARGET_HAS:
#             continue
#         with rasterio.open(tif) as src:
#             data = src.read(1)
#             porosity = np.vectorize(lambda x: porosity_map.get(x, np.nan))(data)
#             porosity[data == src.nodata] = np.nan
#             soil_grids[ha_num] = {
#                 "grid": porosity,
#                 "meta": src.meta,
#                 "transform": src.transform,
#                 "crs": src.crs,
#                 "shape": data.shape,
#             }
#     return soil_grids

# def process_event(row, dem_path, ensemble, porosity_grids, dem_cached, TARGET_HAS):
#     try:
#         year, month, day = int(row['start_year']), int(row['start_month']), int(row['start_day'])
#         event_num = int(row['event_num'])
#         start_index = int(row['start_indices'])

#         ha_num = extract_ha_num(dem_path)
#         if ha_num is None:
#             logging.warning(f"[DEM Parse Fail] Could not extract HA from {dem_path}")
#             return
#         if ha_num not in TARGET_HAS:
#             return

#         ensemble_nc_dir = os.path.join(NETCDF_DIR, ensemble)
#         nc_path = find_matching_netcdf(ensemble_nc_dir, year, month, day)
#         if not nc_path:
#             logging.error(f"[NetCDF Missing] Event {event_num} {year}-{month}-{day} in {ensemble}")
#             return

#         ds = xr.open_dataset(nc_path)
#         index = (start_index - 1) // 24
#         if ha_num not in dem_cached:
#             logging.error(f"[DEM Missing] {ha_num} not found in dem_cached")
#             ds.close()
#             return

#         bounds_geom = dem_cached[ha_num]["bounds"]
#         dem_mask = dem_cached[ha_num]["mask"]
#         dem_info = dem_cached[ha_num]

#         event_base = f"soil_{ha_num}_{event_num}_{year}_{month}_{day}_{ensemble}"

#         # Save original gravitational TIF
#         grav_tif = os.path.join(OUTPUT_ORIG_TIF_DIR, ensemble, f"{event_base}_grav.tif")
#         export_gravitational(ds.isel(time=index), 'moisture_content_of_soil_layer', grav_tif, bounds_geom)

#         # Resample to 30m
#         grav_30m_array, grav_30m_meta = resample_to_30m_array(grav_tif, dem_info)

#         # Volumetric output paths
#         vol_tif = os.path.join(OUTPUT_VOL_DIR, ensemble, "tif", ha_num, f"{event_base}_vol_30m.tif")
#         vol_asc = os.path.join(OUTPUT_VOL_DIR, ensemble, "asc", ha_num, f"{event_base}_vol_30m.asc")

#         if ha_num in porosity_grids:
#             poro_array = porosity_grids[ha_num]["grid"]
#             convert_to_volumetric_array(grav_30m_array, grav_30m_meta, vol_tif, vol_asc, poro_array, dem_mask)
#         else:
#             logging.error(f"[Porosity Missing] {ha_num} for {event_base}")

#         ds.close()

#     except Exception as e:
#         logging.error(f"[Processing Error] Event {row.get('event_num', 'unknown')}: {e}")

# # ==========================
# # Run Pipeline
# # ==========================
# if __name__ == "__main__":
#     parser = argparse.ArgumentParser(description="Run soil moisture pipeline for specific HA_NUM(s).")
#     parser.add_argument("target_has", nargs="+", help="One or more HA_NUM values to process (space-separated). Examples: 27 27_a 54_b 39_d")
#     args = parser.parse_args()
#     TARGET_HAS = set(args.target_has)

#     total_start = time.time()

#     # --------------------------
#     # Logging Setup
#     # --------------------------
#     log_time = datetime.now().strftime("%Y%m%d_%H%M%S")
#     target_str = "_".join(sorted(TARGET_HAS))
#     os.makedirs(LOG_DIR, exist_ok=True)
#     LOG_FILE = os.path.join(LOG_DIR, f"soil_pipeline_{target_str}_{log_time}.txt")
#     logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

#     porosity_grids = load_porosity_grids(POROSITY_CSV, SOIL_TEXTURE_DIR, TARGET_HAS)

#     gdf = load_catchments(SHAPEFILE_PATH)
#     # keep HA_NAME exactly as in shapefile but create a compact token for filename matching (removes spaces)
#     gdf["HA_NAME_NORM"] = gdf["HA_NAME"].str.replace(" ", "", regex=False)

#     # Filter catchments by exact HA_NUM tokens (these may include suffixes like _a)
#     target_catchments = gdf[gdf["HA_NUM"].astype(str).isin(TARGET_HAS)]
#     if target_catchments.empty:
#         logging.error(f"No catchments found for HA_NUM(s): {', '.join(TARGET_HAS)}")
#         sys.exit()

#     target_names_norm = target_catchments["HA_NAME_NORM"].unique().tolist()
#     logging.info(f"Normalized HA_NAMEs for selected HA_NUMs: {target_names_norm}")

#     all_csv_files = glob.glob(os.path.join(CSV_DIR, "*.csv"))

#     # Token-aware CSV selection (avoid substring collisions such as GreatOuse vs Ouse(Yorkshire))
#     csv_files = []
#     for f in all_csv_files:
#         bname = os.path.basename(f)
#         for name in target_names_norm:
#             # match whole token: either start or preceded by '_' and either end or followed by '_'
#             pattern = r'(^|_)' + re.escape(name) + r'(_|$)'
#             if re.search(pattern, bname):
#                 csv_files.append(f)
#                 break

#     if not csv_files:
#         logging.warning("No matching CSV files found.")
#         sys.exit()

#     ensembles = sorted([
#         d for d in os.listdir(NETCDF_DIR)
#         if os.path.isdir(os.path.join(NETCDF_DIR, d))
#     ])

#     # Find DEM files that correspond to any of the requested HA identifiers (full token match)
#     dem_files = []
#     for f in glob.glob(os.path.join(DEM_DIR, "*.tif")):
#         ha = extract_ha_num(f)
#         if ha and ha in TARGET_HAS:
#             dem_files.append(f)

#     dem_cached = {}
#     for dem_path in dem_files:
#         ha_num = extract_ha_num(dem_path)
#         if ha_num not in TARGET_HAS:
#             continue
#         with rasterio.open(dem_path) as dem:
#             dem_cached[ha_num] = {
#                 "bounds": box(*dem.bounds),
#                 "transform": dem.transform,
#                 "crs": dem.crs,
#                 "width": dem.width,
#                 "height": dem.height,
#                 "mask": dem.read(1)
#             }

#     for csv_path in csv_files:
#         csv_name = os.path.basename(csv_path)

#         # Ensemble extraction: look for _NN_ in filename and map to Ens_NN
#         match = re.search(r'_(\d{2})_', csv_name)
#         if not match:
#             logging.warning(f"[CSV Skipped] Could not identify ensemble for {csv_name}")
#             continue

#         ensemble = f"Ens_{match.group(1)}"
#         if ensemble not in ensembles:
#             logging.warning(f"[CSV Skipped] Ensemble {ensemble} not in available ensemble dirs.")
#             continue

#         logging.info(f"[CSV Matched] {csv_name} | Ensemble: {ensemble}")
#         events_df = pd.read_csv(csv_path)

#         # Create base directories once per catchment
#         os.makedirs(os.path.join(OUTPUT_ORIG_TIF_DIR, ensemble), exist_ok=True)
#         for dem_path in dem_files:
#             ha_num = extract_ha_num(dem_path)
#             if ha_num not in TARGET_HAS:
#                 continue
#             os.makedirs(os.path.join(OUTPUT_VOL_DIR, ensemble, "tif", ha_num), exist_ok=True)
#             os.makedirs(os.path.join(OUTPUT_VOL_DIR, ensemble, "asc", ha_num), exist_ok=True)

#         if DRY_RUN:
#             for dem_path in dem_files:
#                 ha_num = extract_ha_num(dem_path)
#                 if ha_num not in TARGET_HAS:
#                     continue
#                 for _, row in events_df.iterrows():
#                     try:
#                         year = int(row['start_year'])
#                         month = int(row['start_month'])
#                         day = int(row['start_day'])
#                         event_num = int(row['event_num'])
#                         event_base = f"soil_{ha_num}_{event_num}_{year}_{month}_{day}_{ensemble}"
#                         print(f"INFO {csv_name} | DEM {ha_num}")
#                         print(f"     • Grav TIF: {os.path.join(OUTPUT_ORIG_TIF_DIR, ensemble, f'{event_base}_grav.tif')}")
#                         print(f"     • Vol TIF:  {os.path.join(OUTPUT_VOL_DIR, ensemble, 'tif', ha_num, f'{event_base}_vol_30m.tif')}")
#                         print(f"     • Vol ASC:  {os.path.join(OUTPUT_VOL_DIR, ensemble, 'asc', ha_num, f'{event_base}_vol_30m.asc')}")
#                     except Exception as e:
#                         print(f"Skipped row due to error: {e}")
#         else:
#             with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
#                 futures = []
#                 for dem_path in dem_files:
#                     ha_num = extract_ha_num(dem_path)
#                     if ha_num not in TARGET_HAS:
#                         continue
#                     for _, row in events_df.iterrows():
#                         futures.append(
#                             executor.submit(
#                                 process_event,
#                                 row,
#                                 dem_path,
#                                 ensemble,
#                                 porosity_grids,
#                                 dem_cached,
#                                 TARGET_HAS
#                             )
#                         )
#                 for future in as_completed(futures):
#                     pass

#     logging.info(f"[Pipeline Finished] Total time: {time.time() - total_start:.2f}s")


In [62]:
#!/usr/bin/env python3
import os
import re
import glob
import rasterio
import xarray as xr
import pandas as pd
import numpy as np
import logging
from shapely.geometry import box
from datetime import datetime
from rasterio.warp import reproject, Resampling
from concurrent.futures import ProcessPoolExecutor, as_completed
import time
import geopandas as gpd
import sys
import argparse

# --------------------------
# Configuration
# --------------------------
CRS_TARGET = "EPSG:27700"
DEM_DIR = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/Model_builds/Pluvial/v4/dem/"
CSV_DIR = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/UKCP_rainfall_events/fixed_threshold_30mm_with_volume/"
NETCDF_DIR = "/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/"
SHAPEFILE_PATH = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/CEH_catchments/CEH_IHU_with_coastline/hyd_areas_GB_with_subcatchments.shp"
SOIL_TEXTURE_DIR = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/Model_builds/Pluvial/v4/soil_texture/"
POROSITY_CSV = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/SoilTexture/USDA_soil_texture_effective_porosity.csv"

OUTPUT_ORIG_TIF_DIR = "/scratch/hydro4/users/kv25483/FutureFlood/Data/UKCP_soil_moisture_events/"

LOG_DIR = "/scratch/hydro4/users/kv25483/FutureFlood/Scripts/Logging/"

DRY_RUN = True
MAX_WORKERS = min(8, os.cpu_count())

In [5]:
# --------------------------
# Helper: robust HA extraction
# --------------------------
def extract_ha_num(path_or_name):
    """
    Extract full HA identifier from a filename or basename.
    Matches:
      - 27
      - 27_a
      - 54_d
      - 33_b
    Returns the matched token (string) or None if not found.
    """
    base = os.path.splitext(os.path.basename(path_or_name))[0]
    # look for a trailing token that starts with digits and may have a _suffix (letters or digits)
    m = re.search(r'(\d+(?:_[A-Za-z0-9]+)?)$', base)
    return m.group(1) if m else None

# --------------------------
# Utility Functions
# --------------------------
def load_catchments(shapefile_path):
    return gpd.read_file(shapefile_path)

def find_matching_netcdf(netcdf_dir, year, month, day):
    target_date_int = int(f"{year:04d}{month:02d}{day:02d}")
    for nc_file in glob.glob(os.path.join(netcdf_dir, "*.nc")):
        match = re.search(r"_(\d{8})-(\d{8})_", os.path.basename(nc_file))
        if match and int(match.group(1)) <= target_date_int <= int(match.group(2)):
            return nc_file
    logging.warning(f"[NetCDF Search Fail] No match for date {target_date_int} in {netcdf_dir}")
    return None

def export_gravitational(ds, var, out_path, bounds):
    if not ds.rio.crs:
        ds = ds.rio.write_crs(CRS_TARGET)
    clipped = ds[[var]].rio.clip_box(
        minx=bounds.bounds[0],
        miny=bounds.bounds[1],
        maxx=bounds.bounds[2],
        maxy=bounds.bounds[3]
    )
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    clipped[var].rio.to_raster(out_path, compress="LZW", tiled=True)


def load_porosity_grids(POROSITY_CSV, SOIL_TEXTURE_DIR, TARGET_HAS):
    df = pd.read_csv(POROSITY_CSV)
    porosity_map = df.set_index("USDA_soil")["Effective_porosity"].to_dict()
    soil_grids = {}
    for tif in glob.glob(os.path.join(SOIL_TEXTURE_DIR, "*.tif")):
        ha_num = extract_ha_num(tif)
        if ha_num is None:
            continue
        if ha_num not in TARGET_HAS:
            continue
        with rasterio.open(tif) as src:
            data = src.read(1)
            porosity = np.vectorize(lambda x: porosity_map.get(x, np.nan))(data)
            porosity[data == src.nodata] = np.nan
            soil_grids[ha_num] = {
                "grid": porosity,
                "meta": src.meta,
                "transform": src.transform,
                "crs": src.crs,
                "shape": data.shape,
            }
    return soil_grids

def process_event(row, dem_path, ensemble, porosity_grids, dem_cached, TARGET_HAS):
    try:
        year, month, day = int(row['start_year']), int(row['start_month']), int(row['start_day'])
        event_num = int(row['event_num'])
        start_index = int(row['start_indices'])

        ha_num = extract_ha_num(dem_path)
        if ha_num is None:
            logging.warning(f"[DEM Parse Fail] Could not extract HA from {dem_path}")
            return
        if ha_num not in TARGET_HAS:
            return

        ensemble_nc_dir = os.path.join(NETCDF_DIR, ensemble)
        nc_path = find_matching_netcdf(ensemble_nc_dir, year, month, day)
        if not nc_path:
            logging.error(f"[NetCDF Missing] Event {event_num} {year}-{month}-{day} in {ensemble}")
            return

        ds = xr.open_dataset(nc_path)
        index = (start_index - 1) // 24
        if ha_num not in dem_cached:
            logging.error(f"[DEM Missing] {ha_num} not found in dem_cached")
            ds.close()
            return

        bounds_geom = dem_cached[ha_num]["bounds"]
        dem_mask = dem_cached[ha_num]["mask"]
        dem_info = dem_cached[ha_num]

        event_base = f"soil_{ha_num}_{event_num}_{year}_{month}_{day}_{ensemble}"

        # Save original gravitational TIF
        grav_tif = os.path.join(OUTPUT_ORIG_TIF_DIR, ensemble, f"{event_base}_grav.tif")
        export_gravitational(ds.isel(time=index), 'moisture_content_of_soil_layer', grav_tif, bounds_geom)

        # Volumetric output paths
#         vol_tif = os.path.join(OUTPUT_VOL_DIR, ensemble, "tif", ha_num, f"{event_base}_vol_30m.tif")

#         if ha_num in porosity_grids:
#             poro_array = porosity_grids[ha_num]["grid"]
#             convert_to_volumetric_array(grav_30m_array, grav_30m_meta, vol_tif, vol_asc, poro_array, dem_mask)
#         else:
#             logging.error(f"[Porosity Missing] {ha_num} for {event_base}")

        ds.close()

    except Exception as e:
        logging.error(f"[Processing Error] Event {row.get('event_num', 'unknown')}: {e}")

In [63]:
# ==========================
# Run Pipeline
# ==========================
def run_pipeline(TARGET_HAS):

    # --------------------------
    # Logging Setup
    # --------------------------
    log_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    target_str = "_".join(sorted(TARGET_HAS))
    os.makedirs(LOG_DIR, exist_ok=True)
    LOG_FILE = os.path.join(LOG_DIR, f"soil_pipeline_{target_str}_{log_time}.txt")
    logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

    porosity_grids = load_porosity_grids(POROSITY_CSV, SOIL_TEXTURE_DIR, TARGET_HAS)

    gdf = load_catchments(SHAPEFILE_PATH)
    # keep HA_NAME exactly as in shapefile but create a compact token for filename matching (removes spaces)
    gdf["HA_NAME_NORM"] = gdf["HA_NAME"].str.replace(" ", "", regex=False)
    

    # Filter catchments by exact HA_NUM tokens (these may include suffixes like _a)
    target_catchments = gdf[gdf["HA_NUM"].astype(str).isin(TARGET_HAS)]
    if target_catchments.empty:
        logging.error(f"No catchments found for HA_NUM(s): {', '.join(TARGET_HAS)}")
        sys.exit()

    target_names_norm = target_catchments["HA_NAME_NORM"].unique().tolist()
    logging.info(f"Normalized HA_NAMEs for selected HA_NUMs: {target_names_norm}")

    all_csv_files = glob.glob(os.path.join(CSV_DIR, "*.csv"))

    # Token-aware CSV selection (avoid substring collisions such as GreatOuse vs Ouse(Yorkshire))
    csv_files = []
    for f in all_csv_files:
        bname = os.path.basename(f)
        for name in target_names_norm:
            # match whole token: either start or preceded by '_' and either end or followed by '_'
            pattern = r'(^|_)' + re.escape(name) + r'(_|$)'
            if re.search(pattern, bname):
                csv_files.append(f)
                break

    if not csv_files:
        logging.warning("No matching CSV files found.")
        sys.exit()

    ensembles = sorted([
        d for d in os.listdir(NETCDF_DIR)
        if os.path.isdir(os.path.join(NETCDF_DIR, d))])

    # Find DEM files that correspond to any of the requested HA identifiers (full token match)
    dem_files = []
    for f in glob.glob(os.path.join(DEM_DIR, "*.tif")):
        ha = extract_ha_num(f)
        if ha and ha in TARGET_HAS:
            dem_files.append(f)
    print(dem_files)

    dem_cached = {}
    for dem_path in dem_files:
        ha_num = extract_ha_num(dem_path)
        if ha_num not in TARGET_HAS:
            continue
        with rasterio.open(dem_path) as dem:
            dem_cached[ha_num] = {
                "bounds": box(*dem.bounds),
                "transform": dem.transform,
                "crs": dem.crs,
                "width": dem.width,
                "height": dem.height,
                "mask": dem.read(1)
            }

    for csv_path in csv_files[:1]:
        csv_name = os.path.basename(csv_path)

        # Ensemble extraction: look for _NN_ in filename and map to Ens_NN
        match = re.search(r'_(\d{2})_', csv_name)
        print(csv_name, match)
        if not match:
            logging.warning(f"[CSV Skipped] Could not identify ensemble for {csv_name}")
            continue

        ensemble = f"Ens_{match.group(1)}"
        if ensemble not in ensembles:
            logging.warning(f"[CSV Skipped] Ensemble {ensemble} not in available ensemble dirs.")
            continue

        logging.info(f"[CSV Matched] {csv_name} | Ensemble: {ensemble}")
        events_df = pd.read_csv(csv_path)
        print(events_df.head())

        # Create base directories once per catchment
        os.makedirs(os.path.join(OUTPUT_ORIG_TIF_DIR, ensemble), exist_ok=True)
#         for dem_path in dem_files:
#             ha_num = extract_ha_num(dem_path)
#             if ha_num not in TARGET_HAS:
#                 continue
#             os.makedirs(os.path.join(OUTPUT_VOL_DIR, ensemble, "tif", ha_num), exist_ok=True)
#             os.makedirs(os.path.join(OUTPUT_VOL_DIR, ensemble, "asc", ha_num), exist_ok=True)

        if DRY_RUN:
            for dem_path in dem_files:
                ha_num = extract_ha_num(dem_path)
                print(ha_num)
                if ha_num not in TARGET_HAS:
                    continue
                for _, row in events_df.iterrows():
                    try:
                        year = int(row['start_year'])
                        month = int(row['start_month'])
                        day = int(row['start_day'])
                        event_num = int(row['event_num'])
                        event_base = f"soil_{ha_num}_{event_num}_{year}_{month}_{day}_{ensemble}"
                        print(f"INFO {csv_name} | DEM {ha_num}")
                        print(f"     • Grav TIF: {os.path.join(OUTPUT_ORIG_TIF_DIR, ensemble, f'{event_base}_grav.tif')}")
                    except Exception as e:
                        print(f"Skipped row due to error: {e}")
        else:
            with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = []
                for dem_path in dem_files:
                    ha_num = extract_ha_num(dem_path)
                    if ha_num not in TARGET_HAS:
                        continue
                    for _, row in events_df.iterrows():
                        futures.append(
                            executor.submit(
                                process_event,
                                row,
                                dem_path,
                                ensemble,
                                porosity_grids,
                                dem_cached,
                                TARGET_HAS
                            )
                        )
                for future in as_completed(futures):
                    pass

    logging.info(f"[Pipeline Finished] Total time: {time.time() - total_start:.2f}s")


In [66]:
test = run_pipeline(["23"])

['/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/Model_builds/Pluvial/v4/dem/dem_filled_30m_23.tif']
Tyne(Northumberland)_01_full_events_with_event_nums.csv <re.Match object; span=(20, 24), match='_01_'>
   start_indices  stop_indices  event_durations     peaks  start_year  \
0           6177          6194               17  30.50427        1992   
1           5293          5303               10  32.84921        1994   
2           5384          5411               27  56.25648        1994   
3           6829          6850               21  35.77378        1995   
4           4236          4250               14  39.58595        2001   

   start_month  start_day  start_hour  start_time  stop_time  \
0            8         18           8    195536.5   195553.5   
1            7         11          12    211932.5   211942.5   
2            7         15           7    212023.5   212050.5   
3            9         15          12    222108.5   222129.5   
4            5         27          1